In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


C:\Users\7860s\AppData\Local\Temp\ipykernel_13124\3373574695.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
d:\BE\YTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Ahatesham_Resume.pdf
✓ Loaded 1 pages

Processing: Ahateshm resume (1).pdf
✓ Loaded 2 pages

Total documents loaded: 3


In [3]:
all_pdf_documents


[Document(metadata={'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'creationdate': '2026-06-05T00:29:14+05:30', 'author': 'Arbaz Hundekar', 'source': '..\\data\\pdf\\Ahatesham_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Ahatesham_Resume.pdf', 'file_type': 'pdf'}, page_content='Ahatesham Mopagar Phone: +91 7892466068\nLinkedIn:-www.linkedin.com/in/ahatesham-mopagar-398288224\nGitHub:- https://github.com/Ahatesham121\nEmail: 7860sham@gmail.com\n• Python Programming with Data Structures and Algorithm and Object Oriented Programming Concepts.\n• Deployed Restful web application using FASTAPI as the backend framework and MySQL as the relational database system.\n\uf0b7 Skills: Python, C++, Microsoft Power BI, MySQL, Lang Chain. Lang Graph, Data Structures and Algorithm, \nObject Oriented Programming.\nCybersecurity simulation Deloitte June 2025 – July 2025\n• Completed a job simulation involving reading web activity logs.\n• Supported a client in a cyb

In [4]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 3 documents into 8 chunks

Example chunk:
Content: Ahatesham Mopagar Phone: +91 7892466068
LinkedIn:-www.linkedin.com/in/ahatesham-mopagar-398288224
GitHub:- https://github.com/Ahatesham121
Email: 7860sham@gmail.com
• Python Programming with Data Stru...
Metadata: {'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'creationdate': '2026-06-05T00:29:14+05:30', 'author': 'Arbaz Hundekar', 'source': '..\\data\\pdf\\Ahatesham_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Ahatesham_Resume.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'creationdate': '2026-06-05T00:29:14+05:30', 'author': 'Arbaz Hundekar', 'source': '..\\data\\pdf\\Ahatesham_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Ahatesham_Resume.pdf', 'file_type': 'pdf'}, page_content='Ahatesham Mopagar Phone: +91 7892466068\nLinkedIn:-www.linkedin.com/in/ahatesham-mopagar-398288224\nGitHub:- https://github.com/Ahatesham121\nEmail: 7860sham@gmail.com\n• Python Programming with Data Structures and Algorithm and Object Oriented Programming Concepts.\n• Deployed Restful web application using FASTAPI as the backend framework and MySQL as the relational database system.\n\uf0b7 Skills: Python, C++, Microsoft Power BI, MySQL, Lang Chain. Lang Graph, Data Structures and Algorithm, \nObject Oriented Programming.\nCybersecurity simulation Deloitte June 2025 – July 2025\n• Completed a job simulation involving reading web activity logs.\n• Supported a client in a cyb

In [6]:
###Embedding and Vectors

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1439.29it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\7860s\AppData\Local\Temp\ipykernel_13124\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 8


In [9]:
chunks


[Document(metadata={'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'creationdate': '2026-06-05T00:29:14+05:30', 'author': 'Arbaz Hundekar', 'source': '..\\data\\pdf\\Ahatesham_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Ahatesham_Resume.pdf', 'file_type': 'pdf'}, page_content='Ahatesham Mopagar Phone: +91 7892466068\nLinkedIn:-www.linkedin.com/in/ahatesham-mopagar-398288224\nGitHub:- https://github.com/Ahatesham121\nEmail: 7860sham@gmail.com\n• Python Programming with Data Structures and Algorithm and Object Oriented Programming Concepts.\n• Deployed Restful web application using FASTAPI as the backend framework and MySQL as the relational database system.\n\uf0b7 Skills: Python, C++, Microsoft Power BI, MySQL, Lang Chain. Lang Graph, Data Structures and Algorithm, \nObject Oriented Programming.\nCybersecurity simulation Deloitte June 2025 – July 2025\n• Completed a job simulation involving reading web activity logs.\n• Supported a client in a cyb

In [10]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 8 texts...


Batches: 100%|██████████| 1/1 [00:04<00:00,  4.16s/it]


Generated embeddings with shape: (8, 384)
Adding 8 documents to vector store...
Successfully added 8 documents to vector store
Total documents in collection: 16


In [11]:
###Retriever
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("Cybersecurity Analyst")

Retrieving documents for query: 'Cybersecurity Analyst'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.41it/s]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_c9dce891_4',
  'content': 'Ahatesham Mopagar\nACADEMIC DETAILS\nSECAB Institute of Engineering and Technology, Bijapur 2024\n (CLASS \nXII) \nExcellent Pu Science College 2020\n (CLASS X) Kendriya Vidyalaya Vijayapura 2018\nComputer Science & \nEngineering\nB.E.\n76 %\nKarnataka Secondary \nEducation Examination \nBoard (KSEEB) 68 %\nCBSE 69 %\nSUBJECTS\nTechnical Proficiency Web Application Security,Networking,Python3,Networking Concepts,Network Security,SQL \nProgramming,C++,Android Testing,Artificial Intelligence,CSS,HTML5\nWORK EXPERIENCE\nINTERNSHIPS\nCybersecurity  Analyst \nInternship\nWeb application Pentesting.\xa0\nNetwork Pentesting.\xa0\nSOCIAL ENGINEERING & PHISHING SIMULATION.\xa0\nMalware Analysis,Reverse Engineering & Threat Hunting.\xa0\nCloud Security Audits\nApr 2025\n-\nMay 2025\nCybersecurity  Analyst \nInternship\nPython programing for cyber Security\nExploring wifi SecurityExploring flipper zero in cybersecurity.\nSecurity Information and event manag

In [17]:
rag_retriever.retrieve("Mobile Application Vulnerability")

Retrieving documents for query: 'Mobile Application Vulnerability'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.47it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_36801ae8_2',
  'content': 'PROJECTS\nMobile Application Vulnerability JAN 2024\n• It is a software tool that scans android apk and performs Static Analysis for Vulnerability Assessment generates \nReport after each scan which includes information about threats that apk file can have. The is implemented on local \nhost server mangoDB as database.\n• Skills: Python, MongoDB, MySQL, Lang Graph.\nAI Agent Using Lang Chain and Lang Graph DEC 2024\n• Ai agent cable answering Question by using Natural Language Understanding.\n• Skills: Python with Lang Chain and Lang Graph, OpenAI API Key, SQL Alchemy.\nBuild Fast API App with MySQL Database\n• This project involves the development of a Restful web application using FASTAPI as the \nbackend framework and MySQL as the relational database system.\n• The application is designed to handle CRUD operations (Create, Read, Update, Delete) \nthrough a high-performance, asynchronous API.\n• Skills:- Python, Fast API , Uvicorn, My SQL, SQL 